In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
from scipy import ndimage

In [ ]:
INPUT_FOLDER = "./data/raw"  # Folder containing input .tif files

# Ordered in steps of the processing pipeline
DIRT_THRESHOLD = 50  # Min blob size to keep
RECTANGLE_DENSITY = 0.95  # Min density of "censored" rectangle
EDGE_THRESHOLD = 50  # Distance from edge within which blobs are removed
CHASM_THRESHOLD = 30  # Max gap (pixels) allowed within an island
BORDER_SIZE = 10  # Padding added around the cropped region
OUTPUT_SIZE = 320  # Final output image size (square)

In [ ]:
def convert_to_bw(image):
    if image.mode != 'L':
        image = image.convert('L')
    return np.where(np.array(image) < 128, 0, 255).astype(np.uint8)


def is_rectangle_by_density(slice_obj, region_mask, density_threshold):
    actual_area = np.sum(region_mask)
    h = slice_obj[0].stop - slice_obj[0].start
    w = slice_obj[1].stop - slice_obj[1].start
    return (actual_area / (h * w)) >= density_threshold


def is_near_edge(slice_obj, img_shape, edge_threshold):
    h, w = img_shape
    return (
        slice_obj[0].start < edge_threshold or
        slice_obj[0].stop  > (h - edge_threshold) or
        slice_obj[1].start < edge_threshold or
        slice_obj[1].stop  > (w - edge_threshold)
    )

In [ ]:
def find_longest_island(pixel_counts, chasm_threshold):
    has_content = pixel_counts > 0

    if not np.any(has_content):
        return 0, 0

    indices = np.where(has_content)[0]
    gaps = np.diff(indices)
    chasm_breaks = np.where(gaps > chasm_threshold)[0]

    starts = np.insert(indices[chasm_breaks + 1], 0, indices[0])
    ends = np.append(indices[chasm_breaks], indices[-1])

    lengths = ends - starts
    longest_idx = np.argmax(lengths)

    return starts[longest_idx], ends[longest_idx]

In [ ]:
def island_crop(img_array, chasm_threshold, border_size, output_size):
    binary_content = (img_array == 0)
    row_counts = np.sum(binary_content, axis=1)
    col_counts = np.sum(binary_content, axis=0)

    r_min, r_max = find_longest_island(row_counts, chasm_threshold)
    c_min, c_max = find_longest_island(col_counts, chasm_threshold)

    if r_min == r_max or c_min == c_max:
        return np.ones((output_size, output_size), dtype=np.uint8) * 255

    r_min = max(0, r_min - border_size)
    r_max = min(img_array.shape[0], r_max + border_size)
    c_min = max(0, c_min - border_size)
    c_max = min(img_array.shape[1], c_max + border_size)

    height, width = r_max - r_min, c_max - c_min
    side     = max(height, width)
    center_r = (r_min + r_max) // 2
    center_c = (c_min + c_max) // 2

    sr1, sr2 = center_r - side // 2, center_r + side // 2
    sc1, sc2 = center_c - side // 2, center_c + side // 2

    canvas   = np.ones((side, side), dtype=np.uint8) * 255
    src_r1   = max(0, sr1)
    src_r2 = min(img_array.shape[0], sr2)
    src_c1   = max(0, sc1)
    src_c2 = min(img_array.shape[1], sc2)
    dest_r1  = max(0, -sr1)
    dest_c1 = max(0, -sc1)
    dest_r2  = dest_r1 + (src_r2 - src_r1)
    dest_c2  = dest_c1 + (src_c2 - src_c1)

    canvas[dest_r1:dest_r2, dest_c1:dest_c2] = img_array[src_r1:src_r2, src_c1:src_c2]

    res_img = Image.fromarray(canvas).resize((output_size, output_size), Image.Resampling.LANCZOS)
    return np.where(np.array(res_img) < 128, 0, 255).astype(np.uint8)

In [ ]:
def process_image(image_path, output_dir, current_idx, total_count,
                  chasm_threshold, dirt_threshold, rectangle_density,
                  edge_threshold, border_size, output_size):
    comp_path = output_dir  / (image_path.stem + "_comparison.tif")

    progress_str = f"[{(current_idx / total_count) * 100:6.2f}%] ({current_idx}/{total_count})"

    if comp_path.exists():
        print(f"{progress_str} Skipping: {image_path.name}")
        return

    print(f"{progress_str} Processing: {image_path.name}")

    img_array      = convert_to_bw(Image.open(image_path))
    original_array = img_array.copy()

    # General dirt/rectangle removal
    labeled_array, num_features = ndimage.label(img_array == 0)
    slices = ndimage.find_objects(labeled_array)

    for i, slc in enumerate(slices):
        if slc is None:
            continue
        mask = (labeled_array[slc] == (i + 1))
        if (
            np.sum(mask) < dirt_threshold or
            is_rectangle_by_density(slc, mask, rectangle_density) or
            is_near_edge(slc, img_array.shape, edge_threshold)
        ):
            img_array[slc][mask] = 255

    # 
    sep = np.zeros((img_array.shape[0], 20), dtype=np.uint8)
    Image.fromarray(np.hstack([img_array, sep, original_array])).save(
        comp_path, compression="tiff_deflate"
    )


In [ ]:
input_path = Path(INPUT_FOLDER)
out_path   = input_path / "processed"
out_path.mkdir(parents=True, exist_ok=True)

files = sorted(list(input_path.glob("*.tif*")))
print(f"Found {len(files)} file(s) in '{INPUT_FOLDER}'")

for idx, f in enumerate(files, 1):
    try:
        process_image(
            f, out_path, idx, len(files),
            chasm_threshold   = CHASM_THRESHOLD,
            dirt_threshold    = DIRT_THRESHOLD,
            rectangle_density = RECTANGLE_DENSITY,
            edge_threshold    = EDGE_THRESHOLD,
            border_size       = BORDER_SIZE,
            output_size       = OUTPUT_SIZE,
        )
    except Exception as e:
        print(f"Error on {f.name}: {e}")